# EDA

In [33]:
import pandas as pd
import os

path = "../data/raw" 
products_df = pd.read_csv(f"{path}/product_info.csv")
print(products_df.columns)

df_reviews_1 = pd.read_csv(f"{path}/reviews_0-250.csv", index_col=0, dtype={'author_id': 'str'}, low_memory=False)
df_reviews_2 = pd.read_csv(f"{path}/reviews_250-500.csv", index_col=0, dtype={'author_id': 'str'}, low_memory=False)
df_reviews_3 = pd.read_csv(f"{path}/reviews_500-750.csv", index_col=0, dtype={'author_id': 'str'}, low_memory=False)
df_reviews_4 = pd.read_csv(f"{path}/reviews_750-1250.csv", index_col=0, dtype={'author_id': 'str'}, low_memory=False)
df_reviews_5 = pd.read_csv(f"{path}/reviews_1250-end.csv", index_col=0, dtype={'author_id': 'str'}, low_memory=False)

df_reviews = pd.concat([df_reviews_1,df_reviews_2,df_reviews_3,df_reviews_4,df_reviews_5],axis=0)
print(df_reviews.columns)

Index(['product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count',
       'rating', 'reviews', 'size', 'variation_type', 'variation_value',
       'variation_desc', 'ingredients', 'price_usd', 'value_price_usd',
       'sale_price_usd', 'limited_edition', 'new', 'online_only',
       'out_of_stock', 'sephora_exclusive', 'highlights', 'primary_category',
       'secondary_category', 'tertiary_category', 'child_count',
       'child_max_price', 'child_min_price'],
      dtype='object')
Index(['author_id', 'rating', 'is_recommended', 'helpfulness',
       'total_feedback_count', 'total_neg_feedback_count',
       'total_pos_feedback_count', 'submission_time', 'review_text',
       'review_title', 'skin_tone', 'eye_color', 'skin_type', 'hair_color',
       'product_id', 'product_name', 'brand_name', 'price_usd'],
      dtype='object')


In [34]:
import ast
import pandas as pd

class DataProcessing:

    columns_to_drop = ['variation_desc', 'sale_price_usd', 'value_price_usd', 'child_max_price', 'child_min_price',            
                       'helpfulness','review_title','tertiary_category','highlights','variation_value',            
                       'variation_type', 'size','total_feedback_count','total_neg_feedback_count',   
                       'total_pos_feedback_count','submission_time', 'limited_edition',            
                       'sephora_exclusive','child_count', 'brand_id', 'online_only', 'new', 
                       'out_of_stock', 'reviews', 'is_recommended', 'review_text']

    critical_columns = ['ingredients', 'author_id', 'rating', 
                        'product_name', 'brand_name', 'price_usd','secondary_category']
    
    user_attribute_columns = ['skin_tone', 'eye_color', 'skin_type', 'hair_color']

    water_keywords = ['water', 'aqua', 'hydrosol']

    silicone_keywords = ['cyclopentasiloxane', 'cyclohexasiloxane', 'dimethicone',
                         'trimethicone', 'amodimethicone', 'vinyl dimethicone', 
                         'cetyl dimethicone', 'phenyl trimethicone','silicone']
    
    category_map = {
        "Cosmetics": ["Moisturizers", "Treatments", "Cleansers", "Masks", "Lip Balms & Treatments", "Sunscreen", "Self Tanners"],
        "Eye Care": ["Eye Care"],
        "Random": ["Mini Size", "Value & Gift Sets", "Wellness", "High Tech Tools", "Shop by Concern"]
    }


    def __init__(self, products_df, reviews_df):
        self.products_df = products_df
        self.reviews_df = reviews_df

    
    def cols_to_use(self):
        """
        Returns the columns to use from the products_df and reviews_df DataFrames
        """
        cols_to_use = self.products_df.columns.difference(self.reviews_df.columns)
        cols_to_use = list(cols_to_use)
        cols_to_use.append('product_id')
        return cols_to_use
    

    def merge_dataframes(self):
        """
        Merges the products_df and reviews_df DataFrames on the 'product_id' column and drop unnecessary columns
        """
        cols_to_use = self.cols_to_use()
        self.merged_df = pd.merge(self.reviews_df, self.products_df[cols_to_use], how='outer', on=['product_id', 'product_id'])
        self.merged_df.drop(columns=self.columns_to_drop, inplace=True, axis=1)
        self.merged_df.drop_duplicates(subset=['product_id'], inplace=True)
    

    def nan_handler(self):
        """
        Fills NaN values in the DataFrame with empty strings
        """
        self.merged_df.dropna(subset=self.critical_columns, how='any', inplace=True)

        for col in self.user_attribute_columns:
            self.merged_df[col].fillna('Unknown', inplace=True)
        
        self.merged_df.drop(columns=['primary_category'], axis=1, inplace=True)
    
    
    def ingredients_to_string(self, ingredients_list):
        """
        Convert a string representation of a list into a single string.
        
        Parameters:
            val (str): A string that represents a list, e.g., "['Water, Butylene Glycol, ...']".
            
        Returns:
            str: A single string with all the list items joined by a space.
        """
        try:
            items = ast.literal_eval(ingredients_list)

            if isinstance(items, list):
                return " ".join(items)
            else:
                return ingredients_list
            
        except Exception as e:
            return ingredients_list
    

    def apply_processing(self):
        """
        Applies the processing functions to the specified column of the DataFrame.
        """
        self.merged_df['ingredients_cleaned'] = self.merged_df['ingredients'].apply(self.ingredients_to_string)
        self.merged_df.drop(columns=['ingredients'], axis=1, inplace=True)

    def classify_ingredient(self, ingredients):
        """
        Classifies the principal ingredient of a product as either water-based or silicone-based.
        """

        ingredients = ingredients.lower()
        if any(keyword in ingredients for keyword in self.water_keywords):
            return "Water"
        elif any(keyword in ingredients for keyword in self.silicone_keywords):
            return "Silicone"
        return None

    def water_or_silicone(self):
        """
        Creates two new columns in the DataFrame, one for water-based products and one for silicone-based products.
        return 2 boolean columns
        """
        self.merged_df["Principal_Ingredient"] = self.merged_df['ingredients'].apply(self.classify_ingredient)
    
    
    def category_classification(self):
        """
        Creates a new column in the DataFrame, that classifies the product into a category.
        return 1 column
        """
        self.merged_df["secondary_category"] = self.merged_df["secondary_category"].apply(
        lambda cat: next((group for group, items in self.category_map.items() if cat in items), "Other")
    )


In [42]:
data_processor = DataProcessing(products_df, df_reviews)
data_processor.merge_dataframes()
data_processor.nan_handler()
data_processor.water_or_silicone()
data_processor.category_classification()
data_processor.apply_processing()

processed_df = data_processor.merged_df
processed_df.skin_type.value_counts()

/tmp/ipykernel_27598/2715148213.py:63: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  self.merged_df[col].fillna('Unknown', inplace=True)


skin_type
combination    1099
dry             513
normal          300
oily            208
Unknown         104
Name: count, dtype: int64

In [36]:
processed_df.sample(5)

,author_id,rating,skin_tone,eye_color,skin_type,hair_color,product_id,product_name,brand_name,price_usd,loves_count,secondary_category,Principal_Ingredient,ingredients_cleaned
1060634,933086994,5.0,fair,hazel,combination,Unknown,P503930,Green Tea Hyaluronic Acid Hydrating Eye Serum,innisfree,22.0,6741,Eye Care,Water,"Water/Aqua/Eau, Butylene Glycol, Propanediol, ..."
478582,8494708454,5.0,lightMedium,brown,combination,black,P433521,Charlotte’s Magic Eye Cream with Retinol,Charlotte Tilbury,65.0,31496,Eye Care,Water,"Aqua/Water/Eau, Caprylic/Capric Triglyceride, ..."
1016471,2845777893,5.0,light,brown,combination,blonde,P500138,Polypeptide-121 Future Cream with Peptides and...,Youth To The People,68.0,29088,Cosmetics,Water,"Water/Aqua/Eau, Dicaprylyl Carbonate, Hydrolyz..."
1046586,2755988672,4.0,porcelain,gray,combination,auburn,P502047,Evercalm Skin Zen Trio,REN Clean Skincare,66.0,1448,Random,Water,"Evercalm Gentle Cleansing Milk: Aqua (Water), ..."
232960,35455417734,5.0,Unknown,Unknown,combination,Unknown,P407444,Bamboo Charcoal Detoxifying Soap Bar,Herbivore,14.0,28997,Cosmetics,None,"Helianthus Annuus (Sunflower) Seed Oil, Cocos ..."


In [37]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

class recommendation_engine:

    def __init__(self, df):
        self.df = df

    def recommendation_function(self, 
                                secondary_category, 
                                ingredient_preference,
                                skintone,
                                skintype,
                                n_recommendations=5):
        """
        Recommends products based on the user's preferences and the similarity of the products' attributes.
        """ 
        
        category_df = self.df[self.df['secondary_category'].str.lower() == secondary_category.lower()]
        filtered_df = category_df[category_df['Principal_Ingredient'].str.lower() == ingredient_preference.lower()]

        # 2. Further filter by user attributes.
        
        filtered_df = filtered_df[filtered_df['skin_tone'].str.lower() == skintone.lower()]
        filtered_df = filtered_df[filtered_df['skin_type'].str.lower() == skintype.lower()]
        

        if filtered_df.empty:
            return "No products found with the specified preferences."

        # Reset index to ensure alignment with tfidf_matrix
        filtered_df = filtered_df.reset_index(drop=True)

        filtered_df['composite_text'] = (filtered_df['ingredients_cleaned'] + " " + filtered_df['skin_tone'].fillna('') + " " + 
                                         filtered_df['skin_type'].fillna('') + " " + filtered_df['Principal_Ingredient'].fillna(''))
        
         # 6 Build the TF-IDF matrix on the composite text.
        vectorizer = TfidfVectorizer()
        tfidf_matrix = vectorizer.fit_transform(filtered_df['composite_text'])
        print(tfidf_matrix.shape)

        # 7. Build the user query from provided user attributes.
        
        if ingredient_preference.lower() == "water":
            query_text = "water aqua hydrosol"
        else:
            query_text = "silicone cyclopentasiloxane dimethicone"
        
        for attr in [query_text, skintone, skintype, ingredient_preference]:
            query_text += " " + attr
        
        input_vector = vectorizer.transform([query_text])

        # Get indices of the filtered products (assumes df and tfidf_matrix are aligned)
        valid_indices = filtered_df.index
        
        # Compute cosine similarity scores between the query and the filtered products
        similarity_scores = cosine_similarity(input_vector, tfidf_matrix[valid_indices]).flatten()
        filtered_df['similarity_score'] = similarity_scores
        filtered_df.sort_values(by=['similarity_score', 'rating'], ascending=[False, False])

        return filtered_df.head(n_recommendations)

In [38]:
recommendations = recommendation_engine(processed_df)
suggestions = recommendations.recommendation_function('Cosmetics', 'Water', 'fair', 'dry')
suggestions

(90, 1415)


,author_id,rating,skin_tone,eye_color,skin_type,hair_color,product_id,product_name,brand_name,price_usd,loves_count,secondary_category,Principal_Ingredient,ingredients_cleaned,composite_text,similarity_score
0,1595163715,5.0,fair,blue,dry,brown,P122767,All About Lips,CLINIQUE,28.0,6581,Cosmetics,Water,"Water, Dimethicone, Ethylene/Acrylic Acid Copo...","Water, Dimethicone, Ethylene/Acrylic Acid Copo...",0.052283
1,1081205269,3.0,fair,blue,dry,blonde,P196542,City Block Sheer Oil-Free Daily Face Protector...,CLINIQUE,30.0,7325,Cosmetics,Water,"Titanium Dioxide 7.30%, Zinc Oxide 6.90%Water,...","Titanium Dioxide 7.30%, Zinc Oxide 6.90%Water,...",0.043559
2,11275015291,3.0,fair,brown,dry,brown,P382356,Age Arrest Anti-Wrinkle Cream,Kate Somerville,98.0,6101,Cosmetics,Water,"Water, Neopentyl Glycol Diheptanoate, Caprylic...","Water, Neopentyl Glycol Diheptanoate, Caprylic...",0.034741
3,12062974858,5.0,fair,green,dry,brunette,P384821,Take The Day Off Cleansing Oil Makeup Remover,CLINIQUE,34.5,29640,Cosmetics,Water,"Cetyl Ethylhexanoate, Triethylhexanoin, Peg-20...","Cetyl Ethylhexanoate, Triethylhexanoin, Peg-20...",0.106915
4,22017698995,5.0,fair,hazel,dry,blonde,P395507,Extra Repair Moisture Cream,Bobbi Brown,110.0,3331,Cosmetics,Water,"Water/Aqua/Eau, Butyrospermum Parkii (Shea) Bu...","Water/Aqua/Eau, Butyrospermum Parkii (Shea) Bu...",0.056287
